# Warm-Start QAOA for Max-Cut

This notebook demonstrates the Regularized Warm-Start (RWS) QAOA workflow for the Max-Cut problem:

1. **Define a graph** — create a random regular graph instance.
2. **Optimize the initial state** — solve the classical warm-start relaxation to obtain product-state angles $\theta$.
3. **Run WS-QAOA** — simulate warm-start QAOA using pre-optimized circuit parameters.
4. **Compare** — show improvement over the initial state energy and standard QAOA.
5. **Generate a Qiskit circuit** — build a hardware-ready WS-QAOA circuit.

In [8]:
import numpy as np
import networkx as nx
from qokit.warm_start import WSSolver
from qokit.qaoa_circuit_maxcut import get_ws_qaoa_circuit

## 1. Define the graph instance

In [2]:
n_v = 20
graph_degree = 3
graph_seed = 0
G = nx.random_regular_graph(graph_degree, n_v, seed=graph_seed)

## 2. Optimize the warm-start initial state

Create a `WSSolver` and optimize the product-state angles $\theta$ using the regularized warm-start (RWS) objective with $\lambda = 0.6$.

In [3]:
solver = WSSolver(graph = G, 
                    graph_degree = graph_degree, 
                    graph_seed = graph_seed,
                    )

In [4]:
lamd = 0.6  # regularization hyperparameter
theta, p0_energy = solver.optimize_theta(
    objective='rws',
    optimizer='ADAM',
    trials=100,
    lamd=lamd,
)
print(f"Initial state energy (p=0): {p0_energy:.4f}")

Initial state energy (p=0): 21.2691


## 3. Run WS-QAOA simulation

Retrieve pre-optimized QAOA parameters $(\gamma, \beta)$ for depth $p=2$ and simulate the warm-start QAOA circuit.

In [5]:
gamma, beta = solver.get_ws_qaoa_para(p=2)
ws_qaoa_energy = solver.run_ws_qaoa(gamma=gamma, beta=beta, theta=theta)
print(f"WS-QAOA energy (p=2): {ws_qaoa_energy:.4f}")

WS-QAOA energy (p=2): 25.0196


## 4. Compare with standard QAOA and lower depths

Compare WS-QAOA at $p=1$ and $p=2$ against standard QAOA at $p=2$.

In [6]:
# WS-QAOA at p=1
gamma_p1, beta_p1 = solver.get_ws_qaoa_para(p=1)
ws_energy_p1 = solver.run_ws_qaoa(gamma=gamma_p1, beta=beta_p1, theta=theta)

# Standard QAOA at p=2
std_energy_p2 = solver.run_standard_qaoa(p=2)

print(f"{'Method':<25} {'Energy':>10}")
print("-" * 36)
print(f"{'WS initial state (p=0)':<25} {p0_energy:>10.4f}")
print(f"{'WS-QAOA (p=1)':<25} {ws_energy_p1:>10.4f}")
print(f"{'WS-QAOA (p=2)':<25} {ws_qaoa_energy:>10.4f}")
print(f"{'Standard QAOA (p=2)':<25} {std_energy_p2:>10.4f}")

Method                        Energy
------------------------------------
WS initial state (p=0)       21.2691
WS-QAOA (p=1)                24.0034
WS-QAOA (p=2)                25.0196
Standard QAOA (p=2)          22.1423


## 5. generate qiskit circuit

In [9]:
gamma, beta = solver.get_ws_qaoa_para(p=1)
qc = get_ws_qaoa_circuit(G, gamma, beta, theta, save_statevector=False)
qc.measure_all()
print(qc)

           ┌────────────┐                                               »
   q0_0: ──┤ Ry(6.2832) ├───────────────────────────────────────────────»
           ├───────────┬┘                                               »
   q0_1: ──┤ Ry(2.905) ├────────────────────────────────────────────────»
           ├───────────┴┐                                               »
   q0_2: ──┤ Ry(2.1199) ├───────────────────────────────────────────────»
           ├────────────┤                                               »
   q0_3: ──┤ Ry(1.7274) ├───■──────────────■────────────────────────────»
           ├────────────┤   │              │                            »
   q0_4: ──┤ Ry(5.0519) ├───┼──────────────┼────────────────────────────»
           ├────────────┤   │              │                            »
   q0_5: ──┤ Ry(1.3874) ├───┼──────────────┼──────────────■─────────────»
           ├────────────┤   │              │ZZ(-0.49727)  │             »
   q0_6: ──┤ Ry(1.2159) ├───┼─────────